<a href="https://colab.research.google.com/github/nitindavegit/AI-Powered-Regulatory-Compliance-Checker-for-Contracts/blob/main/Week_1_SQL_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  SQL Interview Preparation Guide
### All Important SQL Concepts Frequently Asked in Technical Interviews
---
> We'll use **SQLite** (built into Python) to run all SQL queries directly in Colab.

## **Topics Covered:**

### ***0. Setup: Database & Tables***
### ***1. Basic SELECT Queries***
### ***2. WHERE Clause and Filtering***
### ***3. Aggregate functions***
### ***4. GROUP BY & HAVING***
### ***5. JOINs***
### ***6. SubQueries***
### ***7. CTEs (Common Table Expressions)***
### ***8. Window Functions***
### ***9. CASE Statements***
### ***10. NULL Handling***
### ***11. String Functions***
### ***12. Union. Intersection, Except***
### ***13. DDL, DML, Constraints***
### ***14. Views***

# ***0. Setup: Database & Tables***

In [18]:
from decorator import dispatch_on
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

def run_sql(query, display=True):
  """Execute SQL and optionally display as DataFrame"""
  try:
    if query.strip().upper().startswith('SELECT'):
      df = pd.read_sql_query(query, conn)
      if display:
        print(df.to_string(index=False))

      return df
    else:
      cursor.execute(query)
      conn.commit()
      print('Query executed successfully')

  except Exception as e:
    print(f' Error: {e}')


def run_sql_multi(queries):
  """Run multiple SQL statements"""
  for q in queries:
    q = q.strip()
    if q:
      run_sql(q,display=False)
  print(' All queries executed')

print('SQLite3 version:', sqlite3.sqlite_version)
print(' Helper functions ready!')

SQLite3 version: 3.37.2
 Helper functions ready!


In [19]:
# create sample table

# Department table
run_sql('''
   CREATE TABLE departments (
      dept_id    INTEGER PRIMARY KEY,
      dept_name  TEXT NOT NULL,
      location   TEXT
   )''')

# Employees Table
run_sql('''
  CREATE TABLE employees (
      emp_id      INTEGER PRIMARY KEY,
      name        TEXT NOT NULL,
      age         INTEGER NOT NULL,
      salary      REAL,
      dept_id     INTEGER,
      hire_date   TEXT,
      manager_id  INTEGER,
      FOREIGN KEY (dept_id) REFERENCES departments(dept_id)
  )''')

# Orders Table
run_sql('''
CREATE TABLE orders (
    order_id    INTEGER PRIMARY KEY,
    emp_id      INTEGER,
    product     TEXT,
    amount      REAL,
    order_date  TEXT,
    status      TEXT
)''')


# Insert sample data
dept_data = [
    (1, 'IT', 'Mumbai'),
    (2, 'HR', 'Delhi'),
    (3, 'Finance', 'Bangalore'),
    (4, 'Marketing', 'Chennai')
]

cursor.executemany('INSERT INTO departments VALUES (?,?,?)', dept_data)

emp_data = [
    (1,  'Max',   28, 75000, 1, '2020-01-15', None),
    (2,  'Lewis',     35, 90000, 1, '2018-06-20', 1),
    (3,  'Charles', 42, 120000,1, '2015-03-10', 1),
    (4,  'Lando',   30, 55000, 2, '2021-09-01', None),
    (5,  'Oscar',     25, 50000, 2, '2022-02-14', 4),
    (6,  'Carlos',   38, 95000, 3, '2017-11-30', None),
    (7,  'Nicki',   33, 80000, 3, '2019-07-22', 6),
    (8,  'Ayrton',    27, 60000, 4, '2021-04-05', None),
    (9,  'Michael',     31, 70000, 4, '2020-08-18', 8),
    (10, 'Sergio',    29, None,  1, '2023-01-10', 3)
]

cursor.executemany('INSERT INTO employees VALUES (?,?,?,?,?,?,?)', emp_data)

order_data = [
    (1,  1, 'Laptop',   85000, '2023-01-15', 'Completed'),
    (2,  2, 'Phone',    45000, '2023-02-20', 'Completed'),
    (3,  1, 'Tablet',   30000, '2023-03-10', 'Pending'),
    (4,  3, 'Monitor',  25000, '2023-01-25', 'Completed'),
    (5,  5, 'Keyboard',  5000, '2023-04-15', 'Cancelled'),
    (6,  2, 'Mouse',     2000, '2023-05-01', 'Completed'),
    (7,  7, 'Laptop',   85000, '2023-06-10', 'Pending'),
    (8,  1, 'Headset',  15000, '2023-07-20', 'Completed'),
    (9,  6, 'Printer',  35000, '2023-08-15', 'Completed'),
    (10, 9, 'Webcam',    8000, '2023-09-01', 'Pending')
]

cursor.executemany('INSERT INTO orders VALUES (?,?,?,?,?,?)', order_data)
conn.commit()

print(' Database setup complete!')
print('Tables created: departments, employees, orders')

Query executed successfully
Query executed successfully
Query executed successfully
 Database setup complete!
Tables created: departments, employees, orders


# ***1. Basic SELECT Queries***

In [20]:
print('All Employees')
run_sql('SELECT * FROM employees')

print('\nSpecific Columns')
run_sql('SELECT name, age, salary FROM employees')

print('\nAliasing')
run_sql('SELECT name AS employee_name, salary AS annual_salary FROM employees')

print('\nDISTINCT')
run_sql('SELECT DISTINCT dept_id FROM employees')

print('\nORDER BY')
run_sql('SELECT name, salary FROM employees ORDER BY salary DESC')

print('\nLIMIT')
run_sql('SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3')

All Employees
 emp_id    name  age   salary  dept_id  hire_date  manager_id
      1     Max   28  75000.0        1 2020-01-15         NaN
      2   Lewis   35  90000.0        1 2018-06-20         1.0
      3 Charles   42 120000.0        1 2015-03-10         1.0
      4   Lando   30  55000.0        2 2021-09-01         NaN
      5   Oscar   25  50000.0        2 2022-02-14         4.0
      6  Carlos   38  95000.0        3 2017-11-30         NaN
      7   Nicki   33  80000.0        3 2019-07-22         6.0
      8  Ayrton   27  60000.0        4 2021-04-05         NaN
      9 Michael   31  70000.0        4 2020-08-18         8.0
     10  Sergio   29      NaN        1 2023-01-10         3.0

Specific Columns
   name  age   salary
    Max   28  75000.0
  Lewis   35  90000.0
Charles   42 120000.0
  Lando   30  55000.0
  Oscar   25  50000.0
 Carlos   38  95000.0
  Nicki   33  80000.0
 Ayrton   27  60000.0
Michael   31  70000.0
 Sergio   29      NaN

Aliasing
employee_name  annual_salary
     

,name,salary
0,Charles,120000.0
1,Carlos,95000.0
2,Lewis,90000.0


# ***2. WHERE Clause and Filtering***

In [21]:
print('Basic WHERE')
run_sql("SELECT name, salary FROM employees WHERE dept_id = 1")

print('\nAND / OR ')
run_sql("SELECT name, age, salary FROM employees WHERE age > 30 AND salary > 80000")

print('\nBETWEEN' )
run_sql("SELECT name, salary FROM employees WHERE salary BETWEEN 60000 AND 90000")

print('\nIN ')
run_sql("SELECT name, dept_id FROM employees WHERE dept_id IN (1, 3)")

print('\nLIKE (pattern matching) ')
run_sql("SELECT name FROM employees WHERE name LIKE 'A%'")
run_sql("SELECT name FROM employees WHERE name LIKE '%e'")

print('\nIS NULL / IS NOT NULL ')
run_sql("SELECT name, salary FROM employees WHERE salary IS NULL")
run_sql("SELECT name, salary FROM employees WHERE salary IS NOT NULL")

Basic WHERE
   name   salary
    Max  75000.0
  Lewis  90000.0
Charles 120000.0
 Sergio      NaN

AND / OR 
   name  age   salary
  Lewis   35  90000.0
Charles   42 120000.0
 Carlos   38  95000.0

BETWEEN
   name  salary
    Max 75000.0
  Lewis 90000.0
  Nicki 80000.0
 Ayrton 60000.0
Michael 70000.0

IN 
   name  dept_id
    Max        1
  Lewis        1
Charles        1
 Carlos        3
  Nicki        3
 Sergio        1

LIKE (pattern matching) 
  name
Ayrton
Empty DataFrame
Columns: [name]
Index: []

IS NULL / IS NOT NULL 
  name salary
Sergio   None
   name   salary
    Max  75000.0
  Lewis  90000.0
Charles 120000.0
  Lando  55000.0
  Oscar  50000.0
 Carlos  95000.0
  Nicki  80000.0
 Ayrton  60000.0
Michael  70000.0


,name,salary
0,Max,75000.0
1,Lewis,90000.0
2,Charles,120000.0
3,Lando,55000.0
4,Oscar,50000.0
5,Carlos,95000.0
6,Nicki,80000.0
7,Ayrton,60000.0
8,Michael,70000.0


# ***3. Aggregate functions***

In [22]:
print('COUNT ')
run_sql('SELECT COUNT(*) AS total_employees FROM employees')
run_sql('SELECT COUNT(salary) AS non_null_salaries FROM employees')

print('\nSUM ')
run_sql('SELECT SUM(salary) AS total_payroll FROM employees')

print('\nAVG ')
run_sql('SELECT ROUND(AVG(salary), 2) AS avg_salary FROM employees')

print('\nMIN / MAX ')
run_sql('SELECT MIN(salary) AS min_salary, MAX(salary) AS max_salary FROM employees')

print('\n Multiple Aggregates ')
run_sql('''
SELECT
    COUNT(*) AS total,
    ROUND(AVG(salary), 0) AS avg_sal,
    MIN(salary) AS min_sal,
    MAX(salary) AS max_sal,
    SUM(salary) AS total_sal
FROM employees
WHERE salary IS NOT NULL
''')

COUNT 
 total_employees
              10
 non_null_salaries
                 9

SUM 
 total_payroll
      695000.0

AVG 
 avg_salary
   77222.22

MIN / MAX 
 min_salary  max_salary
    50000.0    120000.0

 Multiple Aggregates 
 total  avg_sal  min_sal  max_sal  total_sal
     9  77222.0  50000.0 120000.0   695000.0


,total,avg_sal,min_sal,max_sal,total_sal
0,9,77222.0,50000.0,120000.0,695000.0


# ***4. GROUP BY & HAVING***

In [23]:
print('GROUP BY ')
run_sql('''
SELECT dept_id, COUNT(*) AS emp_count, ROUND(AVG(salary),0) AS avg_salary
FROM employees
GROUP BY dept_id
ORDER BY avg_salary DESC
''')

print('\nHAVING (filter groups) ')
# HAVING filters AFTER grouping (vs WHERE which filters before)
run_sql('''
SELECT dept_id, COUNT(*) AS emp_count, ROUND(AVG(salary),0) AS avg_salary
FROM employees
GROUP BY dept_id
HAVING COUNT(*) >= 3
ORDER BY avg_salary DESC
''')

print('\nWHERE + GROUP BY + HAVING (all together) ')
run_sql('''
SELECT dept_id, COUNT(*) AS emp_count, MAX(salary) AS max_salary
FROM employees
WHERE salary IS NOT NULL
GROUP BY dept_id
HAVING MAX(salary) > 70000
ORDER BY max_salary DESC
''')

GROUP BY 
 dept_id  emp_count  avg_salary
       1          4     95000.0
       3          2     87500.0
       4          2     65000.0
       2          2     52500.0

HAVING (filter groups) 
 dept_id  emp_count  avg_salary
       1          4     95000.0

WHERE + GROUP BY + HAVING (all together) 
 dept_id  emp_count  max_salary
       1          3    120000.0
       3          2     95000.0


,dept_id,emp_count,max_salary
0,1,3,120000.0
1,3,2,95000.0


# ***5. JOINs***

In [24]:
print('INNER JOIN (matching rows only) ')
run_sql('''
SELECT e.name, e.salary, d.dept_name, d.location
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
ORDER BY d.dept_name
''')

print('\nLEFT JOIN (all from left + matching from right) ')
run_sql('''
SELECT e.name, o.product, o.amount, o.status
FROM employees e
LEFT JOIN orders o ON e.emp_id = o.emp_id
ORDER BY e.name
''')

print('\nMulti-table JOIN ')
run_sql('''
SELECT e.name, d.dept_name, o.product, o.amount
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
INNER JOIN orders o ON e.emp_id = o.emp_id
ORDER BY e.name
''')

print('\nSELF JOIN (manager-employee relationship) ')
run_sql('''
SELECT e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY manager
''')

INNER JOIN (matching rows only) 
   name   salary dept_name  location
 Carlos  95000.0   Finance Bangalore
  Nicki  80000.0   Finance Bangalore
  Lando  55000.0        HR     Delhi
  Oscar  50000.0        HR     Delhi
    Max  75000.0        IT    Mumbai
  Lewis  90000.0        IT    Mumbai
Charles 120000.0        IT    Mumbai
 Sergio      NaN        IT    Mumbai
 Ayrton  60000.0 Marketing   Chennai
Michael  70000.0 Marketing   Chennai

LEFT JOIN (all from left + matching from right) 
   name  product  amount    status
 Ayrton     None     NaN      None
 Carlos  Printer 35000.0 Completed
Charles  Monitor 25000.0 Completed
  Lando     None     NaN      None
  Lewis    Mouse  2000.0 Completed
  Lewis    Phone 45000.0 Completed
    Max  Headset 15000.0 Completed
    Max   Laptop 85000.0 Completed
    Max   Tablet 30000.0   Pending
Michael   Webcam  8000.0   Pending
  Nicki   Laptop 85000.0   Pending
  Oscar Keyboard  5000.0 Cancelled
 Sergio     None     NaN      None

Multi-table JOIN 
 

,employee,manager
0,Max,None
1,Lando,None
2,Carlos,None
3,Ayrton,None
4,Michael,Ayrton
5,Nicki,Carlos
6,Sergio,Charles
7,Oscar,Lando
8,Lewis,Max
9,Charles,Max


# ***6. SubQueries***

In [25]:
print('Subquery in WHERE ')
# Employees who earn more than average salary
run_sql('''
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC
''')

print('\nSubquery with IN ')
# Employees in departments located in Mumbai
run_sql('''
SELECT name, dept_id
FROM employees
WHERE dept_id IN (SELECT dept_id FROM departments WHERE location = 'Mumbai')
''')

print('\n Correlated Subquery ')
# Employees who earn more than average salary in their own department
run_sql('''
SELECT e.name, e.salary, e.dept_id
FROM employees e
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.dept_id = e.dept_id
)
ORDER BY e.dept_id
''')

print('\nSubquery in FROM (Derived Table) ')
run_sql('''
SELECT dept_id, avg_sal
FROM (
    SELECT dept_id, ROUND(AVG(salary),0) AS avg_sal
    FROM employees
    GROUP BY dept_id
) AS dept_avg
WHERE avg_sal > 70000
''')

Subquery in WHERE 
   name   salary
Charles 120000.0
 Carlos  95000.0
  Lewis  90000.0
  Nicki  80000.0

Subquery with IN 
   name  dept_id
    Max        1
  Lewis        1
Charles        1
 Sergio        1

 Correlated Subquery 
   name   salary  dept_id
Charles 120000.0        1
  Lando  55000.0        2
 Carlos  95000.0        3
Michael  70000.0        4

Subquery in FROM (Derived Table) 
 dept_id  avg_sal
       1  95000.0
       3  87500.0


,dept_id,avg_sal
0,1,95000.0
1,3,87500.0


# ***7. CTEs (Common Table Expressions)***

In [26]:
print('Basic CTE ')
run_sql('''
WITH high_earners AS (
    SELECT emp_id, name, salary, dept_id
    FROM employees
    WHERE salary > 70000
)
SELECT h.name, h.salary, d.dept_name
FROM high_earners h
JOIN departments d ON h.dept_id = d.dept_id
ORDER BY h.salary DESC
''')

print('\nMultiple CTEs ')
run_sql('''
WITH dept_avg AS (
    SELECT dept_id, ROUND(AVG(salary),0) AS avg_salary
    FROM employees
    GROUP BY dept_id
),
dept_info AS (
    SELECT d.dept_id, d.dept_name, da.avg_salary
    FROM departments d
    JOIN dept_avg da ON d.dept_id = da.dept_id
)
SELECT dept_name, avg_salary
FROM dept_info
ORDER BY avg_salary DESC
''')

print('\n Recursive CTE (Employee hierarchy) ')
run_sql('''
WITH RECURSIVE emp_hierarchy AS (
    -- Base case: top-level employees (no manager)
    SELECT emp_id, name, manager_id, 0 AS level
    FROM employees
    WHERE manager_id IS NULL
    UNION ALL
    -- Recursive case
    SELECT e.emp_id, e.name, e.manager_id, eh.level + 1
    FROM employees e
    INNER JOIN emp_hierarchy eh ON e.manager_id = eh.emp_id
)
SELECT emp_id, name, manager_id,
       CASE WHEN level=0 THEN 'Manager' ELSE 'Employee' END AS role,
       level
FROM emp_hierarchy
ORDER BY level, emp_id
''')

Basic CTE 
Query executed successfully

Multiple CTEs 
Query executed successfully

 Recursive CTE (Employee hierarchy) 
Query executed successfully


# ***8. Window Functions***

In [27]:
print('ROW_NUMBER ')
run_sql('''
SELECT name, salary, dept_id,
       ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS row_num
FROM employees
WHERE salary IS NOT NULL
ORDER BY dept_id, row_num
''')

print('\nRANK vs DENSE_RANK ')
run_sql('''
SELECT name, salary,
       RANK() OVER (ORDER BY salary DESC) AS rank,
       DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rank
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary DESC
''')

print('\n NTILE (quartiles) ')
run_sql('''
SELECT name, salary,
       NTILE(4) OVER (ORDER BY salary) AS quartile
FROM employees
WHERE salary IS NOT NULL
''')

print('\nLAG and LEAD ')
run_sql('''
SELECT name, salary,
       LAG(salary)  OVER (ORDER BY salary) AS prev_salary,
       LEAD(salary) OVER (ORDER BY salary) AS next_salary,
       salary - LAG(salary) OVER (ORDER BY salary) AS diff_from_prev
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary
''')

print('\nRunning Total (SUM window) ')
run_sql('''
SELECT name, salary,
       SUM(salary) OVER (ORDER BY salary) AS running_total
FROM employees
WHERE salary IS NOT NULL
ORDER BY salary
''')

print('\nTOP N per Group (Classic Interview Q) ')
run_sql('''
SELECT name, salary, dept_id
FROM (
    SELECT name, salary, dept_id,
           ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS rn
    FROM employees
    WHERE salary IS NOT NULL
)
WHERE rn <= 2
ORDER BY dept_id, salary DESC
''')

ROW_NUMBER 
   name   salary  dept_id  row_num
Charles 120000.0        1        1
  Lewis  90000.0        1        2
    Max  75000.0        1        3
  Lando  55000.0        2        1
  Oscar  50000.0        2        2
 Carlos  95000.0        3        1
  Nicki  80000.0        3        2
Michael  70000.0        4        1
 Ayrton  60000.0        4        2

RANK vs DENSE_RANK 
   name   salary  rank  dense_rank
Charles 120000.0     1           1
 Carlos  95000.0     2           2
  Lewis  90000.0     3           3
  Nicki  80000.0     4           4
    Max  75000.0     5           5
Michael  70000.0     6           6
 Ayrton  60000.0     7           7
  Lando  55000.0     8           8
  Oscar  50000.0     9           9

 NTILE (quartiles) 
   name   salary  quartile
  Oscar  50000.0         1
  Lando  55000.0         1
 Ayrton  60000.0         1
Michael  70000.0         2
    Max  75000.0         2
  Nicki  80000.0         3
  Lewis  90000.0         3
 Carlos  95000.0         4
Cha

,name,salary,dept_id
0,Charles,120000.0,1
1,Lewis,90000.0,1
2,Lando,55000.0,2
3,Oscar,50000.0,2
4,Carlos,95000.0,3
5,Nicki,80000.0,3
6,Michael,70000.0,4
7,Ayrton,60000.0,4


# ***9. CASE Statements***

In [28]:
print('Simple CASE ')
run_sql('''
SELECT name, salary,
       CASE
           WHEN salary >= 100000 THEN 'Senior'
           WHEN salary >= 75000  THEN 'Mid-Level'
           WHEN salary >= 50000  THEN 'Junior'
           ELSE 'Trainee'
       END AS level
FROM employees
ORDER BY salary DESC
''')

print('\nCASE in aggregate (Conditional Count) ')
run_sql('''
SELECT
    dept_id,
    COUNT(*) AS total,
    COUNT(CASE WHEN salary >= 75000 THEN 1 END) AS high_earners,
    COUNT(CASE WHEN salary < 75000  THEN 1 END) AS others
FROM employees
WHERE salary IS NOT NULL
GROUP BY dept_id
''')

print('\nCASE in ORDER BY ')
run_sql('''
SELECT product, status
FROM orders
ORDER BY
    CASE status
        WHEN 'Pending'   THEN 1
        WHEN 'Completed' THEN 2
        WHEN 'Cancelled' THEN 3
    END
''')

Simple CASE 
   name   salary     level
Charles 120000.0    Senior
 Carlos  95000.0 Mid-Level
  Lewis  90000.0 Mid-Level
  Nicki  80000.0 Mid-Level
    Max  75000.0 Mid-Level
Michael  70000.0    Junior
 Ayrton  60000.0    Junior
  Lando  55000.0    Junior
  Oscar  50000.0    Junior
 Sergio      NaN   Trainee

CASE in aggregate (Conditional Count) 
 dept_id  total  high_earners  others
       1      3             3       0
       2      2             0       2
       3      2             2       0
       4      2             0       2

CASE in ORDER BY 
 product    status
  Tablet   Pending
  Laptop   Pending
  Webcam   Pending
  Laptop Completed
   Phone Completed
 Monitor Completed
   Mouse Completed
 Headset Completed
 Printer Completed
Keyboard Cancelled


,product,status
0,Tablet,Pending
1,Laptop,Pending
2,Webcam,Pending
3,Laptop,Completed
4,Phone,Completed
5,Monitor,Completed
6,Mouse,Completed
7,Headset,Completed
8,Printer,Completed
9,Keyboard,Cancelled


# ***10. NULL Handling***

In [29]:
print('COALESCE (return first non-null) ')
run_sql('''
SELECT name, salary,
       COALESCE(salary, 0) AS salary_with_default
FROM employees
ORDER BY salary
''')

print('\n COALESCE for manager name ')
run_sql('''
SELECT e.name,
       COALESCE(m.name, 'No Manager') AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id
''')

print('\nNULLIF (returns NULL if values are equal) ')
run_sql('''
SELECT name, dept_id,
       NULLIF(dept_id, 1) AS dept_not_it
FROM employees
LIMIT 5
''')

print('\nNULL in aggregation ')
run_sql('''
SELECT
    COUNT(*) AS total_rows,
    COUNT(salary) AS non_null_salary,
    COUNT(*) - COUNT(salary) AS null_salary_count
FROM employees
''')

COALESCE (return first non-null) 
   name   salary  salary_with_default
 Sergio      NaN                  0.0
  Oscar  50000.0              50000.0
  Lando  55000.0              55000.0
 Ayrton  60000.0              60000.0
Michael  70000.0              70000.0
    Max  75000.0              75000.0
  Nicki  80000.0              80000.0
  Lewis  90000.0              90000.0
 Carlos  95000.0              95000.0
Charles 120000.0             120000.0

 COALESCE for manager name 
   name    manager
    Max No Manager
  Lewis        Max
Charles        Max
  Lando No Manager
  Oscar      Lando
 Carlos No Manager
  Nicki     Carlos
 Ayrton No Manager
Michael     Ayrton
 Sergio    Charles

NULLIF (returns NULL if values are equal) 
   name  dept_id  dept_not_it
    Max        1          NaN
  Lewis        1          NaN
Charles        1          NaN
  Lando        2          2.0
  Oscar        2          2.0

NULL in aggregation 
 total_rows  non_null_salary  null_salary_count
         10     

,total_rows,non_null_salary,null_salary_count
0,10,9,1


# ***11. String Functions***

In [30]:
run_sql('''
SELECT name,
       UPPER(name)       AS upper_name,
       LOWER(name)       AS lower_name,
       LENGTH(name)      AS name_length,
       SUBSTR(name,1,3)  AS first_3,
       REPLACE(name,'a','@') AS replaced
FROM employees
LIMIT 5
''')

print('\nTRIM / LTRIM / RTRIM ')
run_sql("""
SELECT TRIM('  hello world  ') AS trimmed,
       LTRIM('  left space')   AS ltrimmed,
       RTRIM('right space  ')  AS rtrimmed
""")

print('\nLIKE patterns ')
run_sql("SELECT name FROM employees WHERE name LIKE '%a%'")
run_sql("SELECT name FROM employees WHERE name LIKE '__i%'")

   name upper_name lower_name  name_length first_3 replaced
    Max        MAX        max            3     Max      M@x
  Lewis      LEWIS      lewis            5     Lew    Lewis
Charles    CHARLES    charles            7     Cha  Ch@rles
  Lando      LANDO      lando            5     Lan    L@ndo
  Oscar      OSCAR      oscar            5     Osc    Osc@r

TRIM / LTRIM / RTRIM 
    trimmed   ltrimmed    rtrimmed
hello world left space right space

LIKE patterns 
   name
    Max
Charles
  Lando
  Oscar
 Carlos
 Ayrton
Michael
Empty DataFrame
Columns: [name]
Index: []


,name


# ***12. Union. Intersection, Except***

In [31]:
print('UNION (combine, remove duplicates) ')
run_sql('''
SELECT name, 'employee' AS source FROM employees WHERE dept_id = 1
UNION
SELECT name, 'employee' AS source FROM employees WHERE salary > 80000
ORDER BY name
''')

print('\nUNION ALL (keep duplicates) ')
run_sql('''
SELECT name FROM employees WHERE dept_id = 1
UNION ALL
SELECT name FROM employees WHERE salary > 80000
ORDER BY name
''')

print('\nEXCEPT / INTERSECT ')
run_sql('''
-- Employees in dept 1 but NOT earning > 80000
SELECT name FROM employees WHERE dept_id = 1
EXCEPT
SELECT name FROM employees WHERE salary > 80000
''')

run_sql('''
-- Employees in dept 1 AND earning > 70000
SELECT name FROM employees WHERE dept_id = 1
INTERSECT
SELECT name FROM employees WHERE salary > 70000
''')

UNION (combine, remove duplicates) 
   name   source
 Carlos employee
Charles employee
  Lewis employee
    Max employee
 Sergio employee

UNION ALL (keep duplicates) 
   name
 Carlos
Charles
Charles
  Lewis
  Lewis
    Max
 Sergio

EXCEPT / INTERSECT 
Query executed successfully
Query executed successfully


# ***13. DDL, DML, Constraints***

In [32]:
print('DDL: CREATE TABLE with Constraints ')
run_sql('''
CREATE TABLE IF NOT EXISTS products (
    product_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL UNIQUE,
    price        REAL NOT NULL CHECK(price > 0),
    category     TEXT DEFAULT 'General',
    stock        INTEGER DEFAULT 0
)
''')

print('\nDML: INSERT ')
run_sql("INSERT INTO products (product_name, price, category, stock) VALUES ('Laptop', 85000, 'Electronics', 50)")
run_sql("INSERT INTO products (product_name, price) VALUES ('USB Cable', 299)")

print('\nDML: UPDATE ')
run_sql("UPDATE products SET price = 80000 WHERE product_name = 'Laptop'")

print('\nDML: DELETE ')
run_sql("DELETE FROM products WHERE stock = 0")

run_sql('SELECT * FROM products')

print('\n DDL: ALTER TABLE ')
run_sql('ALTER TABLE products ADD COLUMN discount REAL DEFAULT 0')
run_sql('SELECT * FROM products')

DDL: CREATE TABLE with Constraints 
Query executed successfully

DML: INSERT 
Query executed successfully
Query executed successfully

DML: UPDATE 
Query executed successfully

DML: DELETE 
Query executed successfully
 product_id product_name   price    category  stock
          1       Laptop 80000.0 Electronics     50

 DDL: ALTER TABLE 
Query executed successfully
 product_id product_name   price    category  stock  discount
          1       Laptop 80000.0 Electronics     50       0.0


,product_id,product_name,price,category,stock,discount
0,1,Laptop,80000.0,Electronics,50,0.0


# ***14. Views***

In [33]:
print('CREATE VIEW ')
run_sql('''
CREATE VIEW IF NOT EXISTS employee_dept_view AS
SELECT e.emp_id, e.name, e.salary, d.dept_name, d.location
FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id
''')

print('\nQuery the View ')
run_sql('SELECT * FROM employee_dept_view ORDER BY dept_name')

print('\nView with filtering ')
run_sql("SELECT * FROM employee_dept_view WHERE dept_name = 'IT'")

print('\nDROP VIEW ')
run_sql('DROP VIEW IF EXISTS employee_dept_view')
print('View dropped!')

CREATE VIEW 
Query executed successfully

Query the View 
 emp_id    name   salary dept_name  location
      6  Carlos  95000.0   Finance Bangalore
      7   Nicki  80000.0   Finance Bangalore
      4   Lando  55000.0        HR     Delhi
      5   Oscar  50000.0        HR     Delhi
      1     Max  75000.0        IT    Mumbai
      2   Lewis  90000.0        IT    Mumbai
      3 Charles 120000.0        IT    Mumbai
     10  Sergio      NaN        IT    Mumbai
      8  Ayrton  60000.0 Marketing   Chennai
      9 Michael  70000.0 Marketing   Chennai

View with filtering 
 emp_id    name   salary dept_name location
      1     Max  75000.0        IT   Mumbai
      2   Lewis  90000.0        IT   Mumbai
      3 Charles 120000.0        IT   Mumbai
     10  Sergio      NaN        IT   Mumbai

DROP VIEW 
Query executed successfully
View dropped!
